# CNN 전체 구조

CNN의 전체 구조를 짧고 간단하게 살펴보자

 고양이 사진이 입력 데이터로 들어온다고 가정해보자. 고양이 사진은 224 224 3(컬러니까 RGB)의 shape를 가졌을 것이다. 이걸 64개의 커널이 합성곱을 수행한다. 그래서 64개 필터가 224, 224 shape의 결과묵을 내놓는다. 그럼 이것을 stackㅎ해서 $224 \times 224 \times 64$ 라는 데이터 덩어리로 만든다. 

가로 세로가 왜 그대로일까? -> 풀링이 없으니까. 

이 과정을 통해 64개의 채널이 쌓였다. 입력 데이터는 RGB 색상 정보 3채널만 가졌다면, 여러가지 특징을 담고 있는 64개 채널의 정보 담고 있는 새로운 차원의 데이터가 되었다. 


그럼 여기서 빨간 박스를 지나면? 
풀링이 일어난다. 가로 세로의 길이가 절반으로 뚝 떨어진다. 단, 채널은 그대로 유지된다. 이게 중요하다! ***풀링은 채널의 개수를 건드리지 않는다.***


그리고 다시 conv2 레이어로 간다. (이때의 데이터는 112,112, 64인 상태임)

그럼 또 어떻게 될까? 당연히 필터가 돌아다니면서 또 채널을 늘리겠지!

con2 에서는 더 복잡한 특징을 찾기 위해 128개의 필터를 사용하도록 설계했다. 필터의 갯수가 곧 채널의 갯수이므로 결과물은 128채널이다.  ****이 때 중요한 것은 conv2의 필터의 채널도 64여야 한다. 입력 데이터가 이미 64층이기 때문에 이를 훑는 커널도 64층이어야 한다.***

연산 방식: 64층짜리 필터 하나가 입력 데이터의 64층을 한꺼번에 덮습니다.

계산: 각 층별로 숫자를 곱하고 더한 뒤, 그 64개의 값을 모두 합쳐서 단 하나의 숫자를 만듭니다.

결과: 64층짜리 입력에 64층짜리 필터 하나를 써도, 결과물은 **1층(1개 채널)**이 나옵니다.

위의 2번 과정을 서로 다른 128개의 필터로 반복합니다.1번 필터(두께 64): 입력 데이터를 훑어 1번 특징 지도(채널 1) 생성2번 필터(두께 64): 입력 데이터를 훑어 2번 특징 지도(채널 2) 생성... (128번까지 반복) ...결과: 이렇게 만들어진 128개의 층을 옆으로 쌓으면 **$112 \times 112 \times 128$**이 된다. 

conv5 까지 모두 지나고  7,7,512 feature map이 만들어졌다. (이건 텐서이겠지) 
그럼 이제 fc layer에 넣어줘야 한다. 근데 fc layer는 한 줄로 세워야 읽을 수 있다. 그래서 flatten 한다. 

그리고 FC layer이다. CNN에서 classification하는 단계에 들어온 것이다. (이전 단계는 모두 feature map을 만들기 위한 일련의 과정이었다.)

Classification (뇌): FC Layer 과정입니다. "눈"이 찾아온 수많은 단서(25,088개의 숫자)를 보고, "귀가 뾰족하고 수염이 있으니 이건 고양이야!"라고 최종 판단을 내리는 역할입니다.
(FC인 이유는 feature map 에 담긴 모든 특징을 전부 고려하겠다는 의미이다.)


-> 여기서 한 가지 사실을 알 수 있는데, CNN에서 학습 대상 W는 두 가지이다. 커널의 가중치(feature map 만드는 첫 단계에의 가중치)와 FC layer의 가중치 W이다.

즉, 특징을 어떻게 뽑을 것인가? 와 추출된 특징으로 어떻게 분류할 것인가? 에 대한 가중치를 학습한다. 

전체 학습 흐름 (Backpropagation)우리가 "이건 강아지야"라고 정답을 알려주면, 모델은 마지막 결과부터 거꾸로 올라오며 두 가중치를 수정합니다.FC 레이어 수정: "결론을 낼 때 이 특징의 비중을 너무 높게 잡았네? 가중치 좀 낮추자."커널 수정: "애초에 특징을 뽑을 때 코 모양을 제대로 못 뽑았네? 필터(W) 숫자를 좀 바꿔서 다시 학습하자.!!! 이렇게 진행된다. 